In [1]:
import pandas as pd
import numpy as np
import json
import joblib
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import os
print("✅ All libraries loaded")

C:\Users\anamt\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All libraries loaded


In [5]:
# === CELL 2: Load Data & Train Ingredient Safety Classifier (Random Forest) ===
# Load ingredient database
db = pd.read_csv('data/ingredients_database.csv', encoding='latin-1')
print(f"Loaded {len(db)} ingredients")

# Use the same embedding model everywhere
emb_model = SentenceTransformer('all-MiniLM-L6-v2')  # free, runs on CPU

# FIXED: Only encode the main 385 ingredient names (not aliases)
ingredient_texts = db['ingredient_name'].tolist()

embeddings = emb_model.encode(ingredient_texts, batch_size=32, show_progress_bar=True)

# Train Random Forest
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    embeddings, 
    db['concern_level'], 
    test_size=0.2, 
    random_state=42, 
    stratify=db['concern_level']
)

from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)
clf.fit(X_train, y_train)

# Evaluate
from sklearn.metrics import classification_report, accuracy_score
y_pred = clf.predict(X_test)
print("✅ Random Forest Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Save models
import joblib
import os
os.makedirs('models', exist_ok=True)
joblib.dump(clf, 'models/safety_rf_model.pkl')
joblib.dump(emb_model, 'models/embedding_model.pkl')
print("✅ Models saved to models/")

Loaded 385 ingredients


Batches: 100%|█████████████████████████████████████████████████████████████████████████| 13/13 [00:01<00:00,  7.87it/s]


✅ Random Forest Accuracy: 0.6103896103896104
              precision    recall  f1-score   support

       Avoid       1.00      0.09      0.17        11
     Caution       0.53      0.38      0.44        26
        Safe       0.63      0.90      0.74        40

    accuracy                           0.61        77
   macro avg       0.72      0.46      0.45        77
weighted avg       0.65      0.61      0.56        77

✅ Models saved to models/


In [6]:
# Load products.json
with open('data/products.json', 'r', encoding='utf-8') as f:
    PRODUCTS = json.load(f)

# Flatten all products into a single list with rich text
all_products = []
product_metadata = []
for category, prods in PRODUCTS.items():
    for p in prods:
        text = f"{p['name']} {p.get('brand','')} {p.get('key_ingredients','')} {' '.join(p.get('concerns',[]))} {p.get('category','')}"
        all_products.append(text)
        product_metadata.append(p)

product_embeddings = emb_model.encode(all_products, batch_size=32, show_progress_bar=True)
np.save('models/product_embeddings.npy', product_embeddings)
joblib.dump(product_metadata, 'models/product_metadata.pkl')
print(f"✅ Pre-computed {len(all_products)} product embeddings")

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 4/4 [00:02<00:00,  1.82it/s]


✅ Pre-computed 105 product embeddings


In [7]:
# Load brands.json
with open('data/brands.json', 'r', encoding='utf-8') as f:
    brands_data = json.load(f)
brands = brands_data['brands']

brand_texts = [f"{b['name']} {' '.join(b.get('aliases',[]))} {b.get('note','')}" for b in brands]
brand_embeddings = emb_model.encode(brand_texts, batch_size=32, show_progress_bar=True)

np.save('models/brands_embeddings.npy', brand_embeddings)
joblib.dump(brands, 'models/brands_metadata.pkl')
print(f"✅ Pre-computed {len(brands)} brand embeddings")

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00,  4.23it/s]

✅ Pre-computed 100 brand embeddings
